<a href="https://colab.research.google.com/github/Shaifali07/langgraph_learning/blob/main/Persistence_Langraph_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.8 MB/s eta 0:00:00


In [5]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from pydantic import BaseModel,fields
from operator import add
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.prompts import ChatPromptTemplate

In [7]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [8]:
llm=ChatGroq(model="llama-3.1-8b-instant",
    temperature=0)

In [9]:
class entertainer(TypedDict):
  topic:str
  joke:Annotated[list[str], add]
  joke_explaination:Annotated[list[str], add]


In [14]:
def generate_joke(state:entertainer):
  topic=state["topic"]
  prompt=ChatPromptTemplate.from_messages([("system","you are a joke teller"),("human","Tell me a joke about {topic} in one line")])
  chain=prompt|llm
  joke= chain.invoke({"topic":topic})
  return {"joke":[joke.content]}

In [13]:
def generate_joke_explaination(state:entertainer):
  joke=state["joke"]
  prompt=ChatPromptTemplate.from_messages([("system","you are a joke explainer"),("human","Explain the given  joke {joke}")])
  chain=prompt|llm
  joke_explaination= chain.invoke({"joke":joke})
  return {"joke_explaination":[joke_explaination.content]}

In [29]:
# from langchain_core.runnables import RunnableConfig
graph=StateGraph(entertainer)
graph.add_node("generate_joke",generate_joke)
graph.add_edge(START,"generate_joke")
graph.add_node("generate_joke_explaination",generate_joke_explaination)
graph.add_edge("generate_joke","generate_joke_explaination")

graph.add_edge("generate_joke_explaination",END,)
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)
config:{"configurable": {"thread_id": "1"}}

initial_state={"topic":"pizza"}
final_state=workflow.invoke(initial_state,config)
print(final_state)

{'topic': 'pizza', 'joke': ['Why was the pizza in a bad mood? Because it was feeling a little crusty.'], 'joke_explaination': ['This joke is a play on words, which is a common technique used in puns. Here\'s a breakdown of how it works:\n\n- The setup of the joke is "Why was the pizza in a bad mood?" This is a typical setup for a joke, asking the listener to consider a situation where a pizza might be feeling a certain way.\n- The punchline is "Because it was feeling a little crusty." At first glance, this might seem like a straightforward statement about the pizza\'s texture. However, the word "crusty" has a double meaning here. In addition to referring to the pizza\'s crust, it\'s also an idiomatic expression that means being irritable or in a bad mood.\n\nThe humor in this joke comes from the unexpected twist on the word "crusty." The listener is initially led to think about the pizza\'s texture, but then the punchline subverts that expectation by using the word in a different way. 

In [30]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'pizza', 'joke': ['Why was the pizza in a bad mood? Because it was feeling a little crusty.'], 'joke_explaination': ['This joke is a play on words, which is a common technique used in puns. Here\'s a breakdown of how it works:\n\n- The setup of the joke is "Why was the pizza in a bad mood?" This is a typical setup for a joke, asking the listener to consider a situation where a pizza might be feeling a certain way.\n- The punchline is "Because it was feeling a little crusty." At first glance, this might seem like a straightforward statement about the pizza\'s texture. However, the word "crusty" has a double meaning here. In addition to referring to the pizza\'s crust, it\'s also an idiomatic expression that means being irritable or in a bad mood.\n\nThe humor in this joke comes from the unexpected twist on the word "crusty." The listener is initially led to think about the pizza\'s texture, but then the punchline subverts that expectation by using the word

In [31]:
history = list(workflow.get_state_history(config))
for state in history:
    print(state)


StateSnapshot(values={'topic': 'pizza', 'joke': ['Why was the pizza in a bad mood? Because it was feeling a little crusty.'], 'joke_explaination': ['This joke is a play on words, which is a common technique used in puns. Here\'s a breakdown of how it works:\n\n- The setup of the joke is "Why was the pizza in a bad mood?" This is a typical setup for a joke, asking the listener to consider a situation where a pizza might be feeling a certain way.\n- The punchline is "Because it was feeling a little crusty." At first glance, this might seem like a straightforward statement about the pizza\'s texture. However, the word "crusty" has a double meaning here. In addition to referring to the pizza\'s crust, it\'s also an idiomatic expression that means being irritable or in a bad mood.\n\nThe humor in this joke comes from the unexpected twist on the word "crusty." The listener is initially led to think about the pizza\'s texture, but then the punchline subverts that expectation by using the word